# 04 Event Aggregation + Speed + Counts + Company

Map detections to lanes, split train events, calculate direction/speed, estimate counts and company labels.

In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd() / 'src'))
import pandas as pd

from rail_video_intelligence.pipeline.aggregation import aggregate_events
from rail_video_intelligence.pipeline.company import enrich_company_labels
from rail_video_intelligence.pipeline.config import load_camera_profile, load_pipeline_settings, parse_pipeline_settings
from rail_video_intelligence.pipeline.detection import run_video_detection, attach_detection_times, create_pseudo_track_ids_for_detect_mode
from rail_video_intelligence.pipeline.ingest import extract_video_metadata

VIDEO_PATH = Path('data/raw/camera_01_2026_04_09_233235.MOV')
PIPELINE_CONFIG = Path('configs/pipeline.yaml')
CAMERA_PROFILE = Path('configs/camera_profiles/camera_01.yaml')

In [ ]:
raw = load_pipeline_settings(PIPELINE_CONFIG)
settings = parse_pipeline_settings(raw.get('pipeline', raw))
profile = load_camera_profile(CAMERA_PROFILE)
meta = extract_video_metadata(VIDEO_PATH)

detections, _, stats = run_video_detection(VIDEO_PATH, settings, Path('outputs/v1'), 'nb_events')
attach_detection_times(detections, meta.fps)
if stats.get('mode_used') == 'detect':
    create_pseudo_track_ids_for_detect_mode(detections)

events = aggregate_events(metadata=meta, detections=detections, profile=profile, settings=settings)
events = enrich_company_labels(metadata=meta, events=events, detections=detections)

rows = [event.as_sheet_row() for event in events]
pd.DataFrame(rows)